# 03 - Convergence Analysis: Is the World Catching Up?

This notebook examines whether global life expectancy is converging (countries getting
more similar) or diverging (inequality growing). Two types of convergence are tested:

- **Sigma convergence:** Is the spread (standard deviation) of life expectancy across countries shrinking?
- **Beta convergence:** Are countries that started with lower life expectancy improving faster?

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 100

PROJECT_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
    PROJECT_DIR = os.getcwd()
    if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
        PROJECT_DIR = os.path.dirname(PROJECT_DIR)

df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'historical', 'merged_historical_panel.csv'))
sigma = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'sigma_convergence.csv'))
beta = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'beta_convergence.csv'))

BLUE_ZONE_ISOS = {'USA', 'JPN', 'ITA', 'GRC', 'CRI'}
print('Data loaded successfully')

Data loaded successfully


## 1. Sigma Convergence: Global Spread Over Time

In [2]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Standard deviation over time
ax1.plot(sigma['year'], sigma['le_std'], 'b-', linewidth=2)
ax1.fill_between(sigma['year'], 0, sigma['le_std'], alpha=0.1, color='blue')
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Standard Deviation of LE (years)', fontsize=12)
ax1.set_title('Sigma Convergence: Global Life Expectancy Spread', fontsize=13)

# Annotate start and end
first_valid = sigma[sigma['le_std'].notna()].iloc[0]
last_valid = sigma[sigma['le_std'].notna()].iloc[-1]
ax1.annotate(f'{first_valid["le_std"]:.1f} yr', 
             xy=(first_valid['year'], first_valid['le_std']),
             xytext=(10, 10), textcoords='offset points', fontsize=10,
             arrowprops=dict(arrowstyle='->', color='black'))
ax1.annotate(f'{last_valid["le_std"]:.1f} yr',
             xy=(last_valid['year'], last_valid['le_std']),
             xytext=(-50, 10), textcoords='offset points', fontsize=10,
             arrowprops=dict(arrowstyle='->', color='black'))

# Right: Coefficient of variation
ax2.plot(sigma['year'], sigma['coefficient_of_variation'], 'r-', linewidth=2)
ax2.fill_between(sigma['year'], 0, sigma['coefficient_of_variation'], alpha=0.1, color='red')
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Coefficient of Variation', fontsize=12)
ax2.set_title('Coefficient of Variation (normalized spread)', fontsize=13)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb03_sigma_convergence.png'),
            dpi=150, bbox_inches='tight')
plt.show()

change = first_valid['le_std'] - last_valid['le_std']
pct_change = (change / first_valid['le_std']) * 100
print(f'\nSigma convergence result: SD decreased from {first_valid["le_std"]:.1f} to {last_valid["le_std"]:.1f} years')
print(f'That is a {pct_change:.0f}% reduction in global life expectancy inequality')


Sigma convergence result: SD decreased from 11.4 to 6.4 years
That is a 44% reduction in global life expectancy inequality


/tmp/ipykernel_1567748/2408753145.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Beta Convergence: Do Lagging Countries Catch Up Faster?

In [3]:
# Beta convergence table
print('Beta Convergence by Decade:')
print('(Negative correlation = countries that started lower improved more = convergence)\n')
beta[['decade', 'n_countries', 'avg_gain_years', 'bz_avg_gain', 'non_bz_avg_gain',
      'beta_correlation', 'convergence']]

Beta Convergence by Decade:
(Negative correlation = countries that started lower improved more = convergence)



,decade,n_countries,avg_gain_years,bz_avg_gain,non_bz_avg_gain,beta_correlation,convergence
0,1960s,93,3.517327,2.653215,3.566424,-0.581415,Yes
1,1970s,93,3.414499,3.465776,3.411586,-0.449965,Yes
2,1980s,93,2.652177,2.594590,2.655449,-0.213024,Yes
3,1990s,93,2.150147,1.728015,2.174132,-0.087705,Weak
4,2000s,93,3.147252,2.093400,3.207130,-0.651764,Yes
5,2010s,93,1.461480,0.317176,1.526497,-0.568845,Yes


In [4]:
# Scatter plot for the most recent complete decade
le_data = df[['iso_code', 'year', 'life_expectancy']].dropna(subset=['life_expectancy'])

# 2000s decade as example
start_data = le_data[(le_data['year'] >= 2000) & (le_data['year'] <= 2002)]
start_data = start_data.sort_values('year').groupby('iso_code').first().reset_index()
end_data = le_data[(le_data['year'] >= 2010) & (le_data['year'] <= 2012)]
end_data = end_data.sort_values('year').groupby('iso_code').first().reset_index()

merged = start_data[['iso_code', 'life_expectancy']].merge(
    end_data[['iso_code', 'life_expectancy']], on='iso_code', suffixes=('_2000', '_2010'))
merged['gain'] = merged['life_expectancy_2010'] - merged['life_expectancy_2000']
merged['is_bz'] = merged['iso_code'].isin(BLUE_ZONE_ISOS)

fig, ax = plt.subplots(figsize=(12, 8))

# Non-BZ countries
non_bz = merged[~merged['is_bz']]
ax.scatter(non_bz['life_expectancy_2000'], non_bz['gain'], alpha=0.5, color='steelblue',
           s=40, label='Other countries')

# BZ countries
bz = merged[merged['is_bz']]
ax.scatter(bz['life_expectancy_2000'], bz['gain'], color='red', s=100, zorder=5,
           marker='*', label='Blue Zone countries')
for _, row in bz.iterrows():
    ax.annotate(row['iso_code'], (row['life_expectancy_2000'], row['gain']),
               xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')

# Trend line
slope, intercept, r, p, se = stats.linregress(merged['life_expectancy_2000'], merged['gain'])
x_line = np.linspace(merged['life_expectancy_2000'].min(), merged['life_expectancy_2000'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'k--', alpha=0.5,
        label=f'Trend (r={r:.3f}, p={p:.4f})')

ax.set_xlabel('Life Expectancy in 2000 (years)', fontsize=12)
ax.set_ylabel('LE Gain 2000-2010 (years)', fontsize=12)
ax.set_title('Beta Convergence: 2000s Decade\n(Countries starting lower gained more)', fontsize=13)
ax.legend(fontsize=10)
ax.axhline(y=0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb03_beta_convergence_scatter.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f'\nBeta convergence (2000s): r = {r:.3f}, slope = {slope:.4f}')
print(f'Interpretation: {"Convergence" if r < 0 else "No convergence"} - '
      f'countries starting lower {"gained more" if slope < 0 else "did not gain more"}')


Beta convergence (2000s): r = -0.652, slope = -0.1319
Interpretation: Convergence - countries starting lower gained more


/tmp/ipykernel_1567748/2435475871.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Blue Zone vs Non-Blue Zone Improvement Comparison

In [5]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(beta))
width = 0.35

ax.bar(x - width/2, beta['bz_avg_gain'], width, label='Blue Zone countries', color='#2196F3')
ax.bar(x + width/2, beta['non_bz_avg_gain'], width, label='Non-Blue Zone countries', color='#FF9800')

ax.set_xlabel('Decade', fontsize=12)
ax.set_ylabel('Average LE Gain (years)', fontsize=12)
ax.set_title('Life Expectancy Gains: Blue Zone vs Non-Blue Zone Countries', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(beta['decade'])
ax.legend(fontsize=11)
ax.axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb03_bz_vs_nonbz_gains.png'),
            dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_1567748/3073627628.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Number of Countries in the Dataset Over Time

In [6]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(sigma['year'], sigma['n_countries'], 'g-', linewidth=2)
ax.fill_between(sigma['year'], 0, sigma['n_countries'], alpha=0.1, color='green')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Number of Countries with LE Data', fontsize=12)
ax.set_title('Data Coverage Over Time', fontsize=13)

plt.tight_layout()
plt.show()

print(f'Countries with LE data in 1960: {int(sigma.iloc[0]["n_countries"])}')
print(f'Countries with LE data in 2023: {int(sigma.iloc[-1]["n_countries"])}')

Countries with LE data in 1960: 92
Countries with LE data in 2023: 93


/tmp/ipykernel_1567748/2265748768.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Summary

In [7]:
print('CONVERGENCE ANALYSIS SUMMARY')
print('=' * 50)
print()
print('Sigma Convergence:')
print(f'  - Global LE standard deviation: {first_valid["le_std"]:.1f} -> {last_valid["le_std"]:.1f} years')
print(f'  - Result: {"CONFIRMED" if last_valid["le_std"] < first_valid["le_std"] else "NOT CONFIRMED"}')
print(f'  - The world is becoming more equal in life expectancy')
print()
print('Beta Convergence:')
n_converging = (beta['convergence'] == 'Yes').sum()
print(f'  - {n_converging} out of {len(beta)} decades show beta convergence')
print(f'  - Countries starting with lower LE generally improved faster')
print()
print('Blue Zone Implications:')
print(f'  - Blue Zone countries are being "caught up to" by the global average')
print(f'  - This is not because BZ countries got worse; it is because others improved faster')

CONVERGENCE ANALYSIS SUMMARY

Sigma Convergence:
  - Global LE standard deviation: 11.4 -> 6.4 years
  - Result: CONFIRMED
  - The world is becoming more equal in life expectancy

Beta Convergence:
  - 5 out of 6 decades show beta convergence
  - Countries starting with lower LE generally improved faster

Blue Zone Implications:
  - Blue Zone countries are being "caught up to" by the global average
  - This is not because BZ countries got worse; it is because others improved faster
